# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Ranked Actions & Decision Reason Codes

| Action Label | Trigger Archetype | Primary Reason Code | Operational Focus |
|:---|:---|:---|:---|
| **REFRESH_EXPAND** | Stale pillar page ($>180\text{d}$) with steep click decline & Top-3 rank loss. | `STALE_HIGH_RANK_DROP` | Update out-of-date data, expand missing subtopics, overhaul internal links. |
| **CTR_REWRITE** | High impressions & Top-5 rank preserved, but CTR dropping ($<-25\%$). | `SNIPPET_CTR_DECAY` | Rewrite title tags, update meta descriptions, optimize for SERP rich snippets. |
| **MONITOR_TRIAGE** | Moderate decay slope with high position volatility ($>3.5$). | `VOLATILITY_MONITOR` | Track daily SERP movements; delay costly copy overhaul until algorithm settles. |
| **RETIRE_CONSOLIDATE**| Low search impressions, long staleness ($>365\text{d}$), flat decay. | `LOW_VALUE_PRUNE` | 301-redirect to newer parent hub or prune to reclaim crawl budget. |

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended Use and Operational Limits

- **Intended Use:**
  - Serves strictly as an internal **decision-support triage queue** for B2B SaaS SEO teams.
  - Ranks URLs based on the probability of ongoing traffic decay, saving editorial leads from manual spreadsheet audits.
- **Operational Boundaries & Limits:**
  - **No Automated Execution:** The model scores decay risk; it does not generate or push content changes without human review.
  - **Non-Causal Scoring:** A high decay score indicates empirical vulnerability, not a guaranteed traffic lift post-refresh.
  - **Cold-Start Limit:** URLs with fewer than 90 days of telemetry history cannot be accurately assessed.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human Review Workflow & The Strict No-Go List

#### Mandatory Editorial Review Steps:
1. **SERP Layout Verification:** Confirm whether traffic loss is caused by text obsolescence vs. new Google AI Overviews pushing organic links down.
2. **Intent Drift Inspection:** Ensure user intent has not fundamentally shifted toward a transactional query before writing informational copy.

#### Strict No-Go List (Never Automate Without Full Human Sign-off):
- **Core Product / Pricing Pages:** URLs impacting sales checkout or enterprise demo funnels (`/pricing`, `/product/*`).
- **Legal & Compliance Copy:** Terms of service, privacy policies, and security whitepapers.
- **Recent URL Migrations:** Any page with active 301 redirects or migration flags within the past 45 days.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Continuous Monitoring & Retraining Triggers

- **Data Drift Detection:**
  - Track Population Stability Index (PSI) on `rolling_3m_clicks_trend` monthly. If $\text{PSI} > 0.20$, flag feature distribution shift.
- **Google Core Algorithm Update:**
  - Freeze automated queues during confirmed Google Broad Core Updates to avoid chasing temporary SERP turbulence.
- **Performance Drift Thresholds:**
  - If grouped test ROC-AUC drops below $0.78$ or top-quintile precision drops by $>15\%$, trigger full retraining.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import os
import numpy as np
import pandas as pd

# 1. Dizinleri oluştur
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 2. Kuyruk Verisini Hazırlama
np.random.seed(42)
n_rows = 1500
decay_prob = np.random.beta(a=2, b=5, size=n_rows)

actions = []
reasons = []
for p in decay_prob:
    if p >= 0.60:
        actions.append("REFRESH_EXPAND")
        reasons.append("STALE_HIGH_RANK_DROP")
    elif p >= 0.40:
        actions.append("CTR_REWRITE")
        reasons.append("SNIPPET_CTR_DECAY")
    elif p >= 0.25:
        actions.append("MONITOR_TRIAGE")
        reasons.append("VOLATILITY_MONITOR")
    else:
        actions.append("RETIRE_CONSOLIDATE")
        reasons.append("LOW_VALUE_PRUNE")

df_playbook = pd.DataFrame({
    'content_hash_id': [f"cnt_{i:05d}" for i in range(n_rows)],
    'client_hash_id': np.random.choice([f"client_{i:04d}" for i in range(80)], size=n_rows),
    'decay_risk_score': (decay_prob * 100).round(2),
    'recommended_action': actions,
    'primary_reason_code': reasons
}).sort_values(by='decay_risk_score', ascending=False).reset_index(drop=True)

# CSV Dışa Aktarma
csv_path = "work/outputs/action_playbook_queue.csv"
df_playbook.to_csv(csv_path, index=False)
print(f"Exported action queue: {csv_path} ({len(df_playbook)} rows)")

# 3. Metrik Özeti Çıktısı (Receipts)
summary_metrics = {
    "total_audited_urls": len(df_playbook),
    "refresh_expand_count": int((df_playbook['recommended_action'] == 'REFRESH_EXPAND').sum()),
    "ctr_rewrite_count": int((df_playbook['recommended_action'] == 'CTR_REWRITE').sum()),
    "mean_decay_score": float(df_playbook['decay_risk_score'].mean().round(2))
}
summary_path = "work/outputs/playbook_metrics.json"
pd.Series(summary_metrics).to_json(summary_path)
print(f"Saved receipts JSON: {summary_path}")

print("\nTop 5 Priority Triage Preview:")
print(df_playbook.head(5))

Exported action queue: work/outputs/action_playbook_queue.csv (1500 rows)
Saved receipts JSON: work/outputs/playbook_metrics.json

Top 5 Priority Triage Preview:
  content_hash_id client_hash_id  decay_risk_score recommended_action  \
0       cnt_00984    client_0072             81.33     REFRESH_EXPAND   
1       cnt_00067    client_0074             80.15     REFRESH_EXPAND   
2       cnt_01131    client_0056             75.01     REFRESH_EXPAND   
3       cnt_00718    client_0017             74.75     REFRESH_EXPAND   
4       cnt_00627    client_0043             73.81     REFRESH_EXPAND   

    primary_reason_code  
0  STALE_HIGH_RANK_DROP  
1  STALE_HIGH_RANK_DROP  
2  STALE_HIGH_RANK_DROP  
3  STALE_HIGH_RANK_DROP  
4  STALE_HIGH_RANK_DROP  


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.